# Day3_13 · 회귀 결과 점검

예측 오차의 모양과 입력변수 관계를 확인해 선형회귀의 한계를 찾습니다.

## 이 노트북에서 확인할 내용

- 실제값·예측값·잔차를 한 표에서 확인할 수 있습니다.
- 잔차 그림에서 체계적인 모양을 찾을 수 있습니다.
- 다중공선성과 영향도가 큰 관측을 점검할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-13-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-13-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-13-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-13-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-13-05 · pandas와 numpy 불러오기

In [ ]:
import pandas as pd
import numpy as np

### D3B-13-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-13-07 · 콘크리트 강도자료 읽기

In [ ]:
concrete = pd.read_csv(DATA_DIR / 'concrete_strength.csv')

In [ ]:
concrete.head()

### D3B-13-08 · 입력 열 정하기

In [ ]:
feature_columns = ['cement_kg_m3', 'slag_kg_m3', 'fly_ash_kg_m3', 'water_kg_m3', 'superplasticizer_kg_m3', 'coarse_aggregate_kg_m3', 'fine_aggregate_kg_m3', 'age_days']

In [ ]:
pd.Series(feature_columns)

### D3B-13-09 · 입력표 만들기

In [ ]:
X = concrete[feature_columns]

In [ ]:
X.head()

### D3B-13-10 · 결과 열 만들기

In [ ]:
y = concrete['strength_mpa']

In [ ]:
y.head()

### D3B-13-11 · 통계적 회귀 도구 불러오기

In [ ]:
import statsmodels.api as sm

### D3B-13-12 · 상수항이 있는 입력표 만들기

In [ ]:
X_with_constant = sm.add_constant(X)

In [ ]:
X_with_constant.head()

### D3B-13-13 · 회귀식 맞추기

In [ ]:
diagnostic_model = sm.OLS(y, X_with_constant).fit()

In [ ]:
diagnostic_model

### D3B-13-14 · 예측값 가져오기

In [ ]:
fitted_strength = diagnostic_model.fittedvalues

In [ ]:
fitted_strength.head()

### D3B-13-15 · 잔차 가져오기

In [ ]:
residual = diagnostic_model.resid

In [ ]:
residual.head()

### D3B-13-16 · 진단용 표 만들기

In [ ]:
diagnostic_table = concrete.copy()
diagnostic_table['predicted_mpa'] = fitted_strength
diagnostic_table['residual_mpa'] = residual
diagnostic_table['absolute_residual_mpa'] = residual.abs()

In [ ]:
diagnostic_table[['strength_mpa', 'predicted_mpa', 'residual_mpa']].head().round(2)

## 실제값과 예측값의 모양 확인하기

### D3B-13-17 · 실제값과 예측값 그리기

In [ ]:
sns.scatterplot(data=diagnostic_table, x='strength_mpa', y='predicted_mpa', alpha=0.5)
limits = [diagnostic_table['strength_mpa'].min(), diagnostic_table['strength_mpa'].max()]
plt.plot(limits, limits, color='red', linestyle='--')
plt.xlabel('Actual strength (MPa)')
plt.ylabel('Predicted strength (MPa)')
plt.show()

### D3B-13-18 · 예측값과 잔차 그리기

In [ ]:
sns.scatterplot(data=diagnostic_table, x='predicted_mpa', y='residual_mpa', alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted strength (MPa)')
plt.ylabel('Residual (MPa)')
plt.show()

### D3B-13-19 · 잔차 분포 그리기

In [ ]:
sns.histplot(diagnostic_table['residual_mpa'], bins=35, kde=True)
plt.xlabel('Residual (MPa)')
plt.show()

### D3B-13-20 · 오차가 큰 행 찾기

In [ ]:
largest_errors = diagnostic_table.nlargest(10, 'absolute_residual_mpa')

In [ ]:
largest_errors[['age_days', 'cement_kg_m3', 'water_kg_m3', 'strength_mpa', 'predicted_mpa', 'residual_mpa']].round(2)

### D3B-13-21 · 재령 구간 만들기

In [ ]:
diagnostic_table['age_group'] = pd.cut(diagnostic_table['age_days'], bins=[0, 7, 28, 90, 365], include_lowest=True)

In [ ]:
diagnostic_table[['age_days', 'age_group']].head()

### D3B-13-22 · 재령 구간별 잔차 요약

In [ ]:
residual_by_age = diagnostic_table.groupby('age_group', observed=True)['residual_mpa'].agg(['count', 'mean', 'median', 'std'])

In [ ]:
residual_by_age.round(2)

### D3B-13-23 · 재령 구간별 잔차 상자그림

In [ ]:
sns.boxplot(data=diagnostic_table, x='age_group', y='residual_mpa')
plt.axhline(0, color='red', linestyle='--')
plt.xticks(rotation=20)
plt.show()

잔차가 0 주변에 무작위로 흩어지지 않고 곡선이나 부채꼴 모양을 만들거나, 특정 재령 구간에서 한쪽으로 치우치면 현재 직선식이 자료의 구조를 충분히 담지 못했을 수 있습니다.

### D3B-13-24 · 입력변수 상관표 만들기

In [ ]:
input_correlation = X.corr()

In [ ]:
input_correlation.round(2)

### D3B-13-25 · 입력변수 상관표를 색으로 보기

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(input_correlation, cmap='vlag', center=0, annot=True, fmt='.2f')
plt.title('Correlations among regression inputs')
plt.show()

### D3B-13-26 · VIF 계산 도구 불러오기

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

### D3B-13-27 · 각 입력열의 VIF 계산

In [ ]:
vif_table = pd.DataFrame({
    'input_column': feature_columns,
    'VIF': [variance_inflation_factor(X.to_numpy(), index) for index in range(X.shape[1])],
})

In [ ]:
vif_table.sort_values('VIF', ascending=False).round(2)

한 입력열이 다른 입력열의 조합과 매우 비슷하면 계수를 따로 분리해 해석하기 어려워집니다. 전문가들은 이를 다중공선성이라고 부르며 VIF와 상관표를 함께 확인합니다.

### D3B-13-28 · 영향도 값 계산

In [ ]:
influence = diagnostic_model.get_influence()
cooks_distance = pd.Series(influence.cooks_distance[0], index=concrete.index, name='cooks_distance')

In [ ]:
cooks_distance.head()

### D3B-13-29 · 영향도가 큰 행 확인

In [ ]:
influential_rows = concrete.assign(cooks_distance=cooks_distance).nlargest(10, 'cooks_distance')

In [ ]:
influential_rows[['age_days', 'cement_kg_m3', 'water_kg_m3', 'strength_mpa', 'cooks_distance']].round(4)

### D3B-13-30 · 확인할 잔차 기준 정하기

In [ ]:
# 20을 15 또는 25로 바꾸어 보세요.
residual_limit = 20

In [ ]:
residual_limit

### D3B-13-31 · 기준보다 큰 잔차 행 선택

In [ ]:
large_residual_rows = diagnostic_table.loc[diagnostic_table['absolute_residual_mpa'] >= residual_limit]

In [ ]:
large_residual_rows[['age_days', 'strength_mpa', 'predicted_mpa', 'residual_mpa']].head(15).round(2)

### D3B-13-32 · 기준보다 큰 잔차 행 수 확인

In [ ]:
len(large_residual_rows)

### D3B-13-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''diagnostic_table을 사용하세요. 기존 셀은 수정하지 마세요. residual_mpa를 predicted_mpa와 age_days에 대해 각각 보여 주는 그림 두 개를 제안하고, 어떤 모양이 보이면 현재 선형회귀를 다시 검토해야 하는지 설명해 주세요. 큰 잔차 행을 자동 삭제하지 마세요.'''.strip()

In [ ]:
codex_request

### D3B-13-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['잔차의 단위가 MPa로 표시되는가?', '잔차가 큰 행을 이유 없이 삭제하지 않는가?', '그래프 모양을 모델 가정과 연결하는가?'], name='확인할 내용')

In [ ]:
codex_checklist